![](https://drive.google.com/uc?export=view&id=1dXA0Ac-6jYiuCGWpVq8pTpnl01vQadR5)*„Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”,
projekt finansowany ze środków Programu Operacyjnego Polska Cyfrowa POPC.03.02.00-00-0001/20*



---

**Przedmiot:** Uczenie Głębokie<br>
**Moduł:** 2 (Podstawy Głębokiego Uczenia)<br>
**Laboratoria:** 3<br>
**Opis laboratoriów:** Zajęcia prezentujące pracę z sieciami neuronowymi w środowisku Python z wykorzystaniem bibliotek TensorFlow oraz PyTorch. Zakres materiału laboratoriów obejmuje tworzenie własnych komponentów i modeli neuronowych, prototypowanie, zaawansowane uczenie własnych modeli neuronowych oraz optymalizację.

---

# Funkcje Pomocnicze

In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import time
import tensorflow as tf
import tensorflow_datasets as tfds

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def print_shapes(name, x, y):
    if torch.is_tensor(x):
        x = x.detach().cpu()
    if torch.is_tensor(y):
        y = x.detach().cpu()
    print(f"{name}: {x.numpy().shape} -> {y.numpy().shape}")

# Budowa modułów neuronowych

## Warstwy

Głębokie sieci neuronowe, charakteryzują się wielokrotnym wykorzystaniem tych samych modułów (**warstw**), bardzo często w sekwencyjnej konfiguracji. Ich schemat działania jest taki sam, a o funkcjonalności decydują **parametry warstwy**. Parametrem może być zarówno liczba neuronów w warstwie **w pełni połączonej** jak i indeks redukowanego wymiaru.

Do najpopularniejszych i najbardziej podstawowych warstw zaliczamy:
- warstwa w pełni połączona (jest to zbiór niezależnych od siebie neuronów wykonanych na tych tych samych wejściach, lub inaczej: **wielokrotna kombinacja liniowa wektora wejściowego**).
- warstwa konwolucyjna,
- warstwa poolingu (operacji podobnej do konwolucji, wykonującej działanie niezawierające parametrów uczalnych).

Biblioteka TensorFlow pozwala na łatwą deklarację wyżej wymienionych warstw:

In [ ]:
# TF
fc = tf.keras.layers.Dense(units=16)
conv = tf.keras.layers.Conv2D(filters=32, kernel_size=3, padding='same')
max_pool = tf.keras.layers.MaxPool2D(pool_size=3, strides=2, padding='same')

# rozmiar tensora x: rozmiar batcha x liczba cech
x = tf.random.normal([4, 8])
print_shapes("fc", x, fc(x))

# rozmiar tensora x: rozmiar batcha x wysokość obrazka x szerokość obrazka x liczba cech (kanałów)
# (BHWC)
x = tf.random.normal([4, 64, 64, 3])
print_shapes("conv", x, conv(x))
print_shapes("max_pool", x, max_pool(x))

fc: (4, 8) -> (4, 16)
conv: (4, 64, 64, 3) -> (4, 64, 64, 32)
max_pool: (4, 64, 64, 3) -> (4, 32, 32, 3)


Analogiczny kod w PyTorch wygląda bardzo podobnie. Różnicą jest konieczność podawania wymiarowości wejść dla oraz odmienna kolejność wymiarów dla danych obrazowych (BHWC vs BCHW).

In [ ]:
# PT
fc = torch.nn.Linear(in_features=8, out_features=16)
conv = torch.nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding="same")
max_pooling = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding='same')

# rozmiar tensora x: rozmiar batcha x liczba cech
x = torch.randn([4, 8])
print_shapes("fc", x, fc(x))

# rozmiar tensora x: rozmiar batcha x liczba cech (kanałów) x wysokość obrazka x szerokość obrazka
# (BCHW)
x = torch.randn([4, 3, 64, 64])
print_shapes("conv", x, conv(x))
print_shapes("max_pool", x, max_pool(x))

fc: (4, 8) -> (4, 8)
conv: (4, 3, 64, 64) -> (4, 3, 64, 64)
max_pool: (4, 3, 64, 64) -> (4, 2, 32, 64)


Deklarując warstwy neuronowe w TensorFlow należy pamiętać o ich procesie inicjalizacji. Inicjalizacja warstw polega na **stworzeniu parametrów uczalnych** i zachowaniu ich w pamięci. Parametry te określają końcowy sposób działania warstwy i mogą być dostrajane za pomocą algorytmu **wstecznej propagacji błędu**.

W TensorFlow istnieją dwa sposoby inicjalizacji warstw:
- automatyczna - warstwa zainicjalizuje się przy jej pierwszym wywołaniu (należy uważać jakie dane podaje się na wejściu, ponieważ będą one determinowały rozmiar parametrów uczalnych),
- ręczna - jawna podanie parametrów działania modelu bez potrzeby wykonania inferencji (odpytania).

In [ ]:
# TF
# deklaracja warstwy w pełni połączonej posiadającej 16 wyjść (neuronów)
fc_auto = tf.keras.layers.Dense(16)

# dane wejściowe to wektor 128 wartości, na których zostanie wykonanych 16 niezależnych kombinacji liniowych
data = tf.zeros([1, 128])

# rezultatem działania jest wektor 16 wartości (dla każdego z elementów w paczce - batchu)
output = fc_auto(data) # <= wagi są inicjalizowane w tym momencie

print('Rozmiar tensora wyjściowego:', output.shape)

Rozmiar tensora wyjściowego: (1, 16)


In [ ]:
# TF
# deklaracja warstwy w pełni połączonej posiadającej 16 wyjść (neuronów)
fc_manual = tf.keras.layers.Dense(16)

# ręczna inicjalizacja warstwy tak, aby mogła być wykorzystywana do przetwarzania wektorów wejściowych o rozmiarze 64
# (None odpowiada rozmiarowi batcha, który może być dowolny)
fc_manual.build([None, 64])

Do parametrów uczalnych można odwołać się przy pomocy własności **trainable_variables**.

In [ ]:
# TF
print('Parametry uczalne warstwy zainicjalizowanej automatycznie:\n', fc_auto.trainable_variables)
print('Parametry uczalne warstwy ręcznie zainicjalizowanej:\n', fc_manual.trainable_variables)

Parametry uczalne warstwy zainicjalizowanej automatycznie:
 [<KerasVariable shape=(128, 16), dtype=float32, path=dense_18/kernel>, <KerasVariable shape=(16,), dtype=float32, path=dense_18/bias>]
Parametry uczalne warstwy ręcznie zainicjalizowanej:
 [<KerasVariable shape=(64, 16), dtype=float32, path=dense_19/kernel>, <KerasVariable shape=(16,), dtype=float32, path=dense_19/bias>]


Z kolei w PyTorch uczalne parametry warstw są tworzone podczas **tworzenia danej warstwy**. Dostęp do parametrów można uzyskać np. poprzez wywołanie metod _parameters_ lub _named_parameters_.

In [ ]:
# PT
fc = torch.nn.Linear(in_features=128, out_features=16)

# Wywołanie "list" ze względu na to, że metoda zwraca generator
# print('Parametry uczalne warstwy:\n', list(fc.parameters()))
print('Parametry uczalne warstwy:\n', list(fc.named_parameters()))

Parametry uczalne warstwy:
 [('weight', Parameter containing:
tensor([[ 0.0652,  0.0344, -0.0849,  ..., -0.0746, -0.0367,  0.0677],
        [ 0.0550, -0.0675, -0.0663,  ...,  0.0127,  0.0566,  0.0245],
        [-0.0647,  0.0112, -0.0610,  ...,  0.0418,  0.0008, -0.0474],
        ...,
        [ 0.0797,  0.0354, -0.0440,  ..., -0.0539, -0.0354, -0.0862],
        [-0.0791, -0.0692, -0.0355,  ..., -0.0538, -0.0064,  0.0656],
        [-0.0805, -0.0377,  0.0127,  ...,  0.0432, -0.0479, -0.0058]],
       requires_grad=True)), ('bias', Parameter containing:
tensor([-0.0705,  0.0859, -0.0635,  0.0811,  0.0763,  0.0307,  0.0617, -0.0720,
        -0.0045,  0.0319, -0.0038, -0.0402,  0.0179,  0.0304,  0.0769,  0.0557],
       requires_grad=True))]


Jak widać na powyższych podglądach, wszystkie warstwy zainicjalizowały parametry (nazwane **kernel** i **bias** w TensorFlow oraz **weight** i **bias** w PyTorch), które są odpowiednikami parametrów kombinacji liniowej:

<br>

$$y_j = \sum_i x_i * w_{ij} + b_i$$

lub stosując zapis macierzowy

$$\boldsymbol{y} = \boldsymbol{W}\boldsymbol{x} + \boldsymbol{b}$$

Warstwy zaimplementowane w bibliotece TensorFlow w module Keras dziedziczą po klasie **Layer**. Aby stworzyć własną warstwę należy zadekladować własną klasę, która dziedziczy po klasie Layer oraz dodać implementację metody *call* (parametrami jest dowolna liczba obiektów). Jeśli będziemy w warstwie korzystać z własnych parametrów uczalnych, powinniśmy je zadekladować w metodzie *build* (parametrem jest rozmiar wejścia).

In [ ]:
# TF
# dziedziczenie po klasie Layer w paczce tf.keras.layers
class MyLayerTF(tf.keras.layers.Layer):
    # dowolny konstruktor, który wewnątrz wywołuje konstruktor klasy nadrzędnej
    def __init__(self, num_weights):
        super().__init__()
        self.num_weights = num_weights

    # funkcja odpowiedzialna za inicjalizację parametrów (jest wywołana podczas pierwszego wywołania lub poczas ręcznej budowy)
    def build(self, input_shape):
        self.w1 = self.add_weight(name="w1", shape=[input_shape[1], self.num_weights], dtype=tf.float32)
        self.w2 = self.add_weight(name="w2", shape=[input_shape[1], self.num_weights], dtype=tf.float32)
        self.w3 = self.add_weight(name="w3", shape=[self.num_weights * 2, self.num_weights], dtype=tf.float32)

    # to przyda nam się później
    def compute_output_shape(self, input_shape):
        return [*input_shape[:-1], self.num_weights]

    # definicja inferencji modelu (nie mylić z metodę __call__!)
    def call(self, inputs):
        x1 = tf.nn.relu(inputs @ self.w1)
        x2 = tf.nn.relu(inputs @ self.w2)
        return tf.nn.relu(tf.concat([x1, x2], axis=1) @ self.w3)

In [ ]:
# TF
# deklaracja
ml = MyLayerTF(16)

# inicjalizacja (pierwsze wywołanie lub metoda build)
# ml.build([None, 4])
ml(tf.random.normal([128, 4]))

<tf.Tensor: shape=(128, 16), dtype=float32, numpy=
array([[0.10360632, 0.        , 0.5498139 , ..., 0.        , 0.550892  ,
        0.        ],
       [0.        , 0.75063735, 0.        , ..., 1.1459877 , 0.89818007,
        0.        ],
       [0.26533064, 0.        , 0.4619648 , ..., 0.0800252 , 0.34066436,
        0.        ],
       ...,
       [0.        , 0.45624614, 0.23759969, ..., 0.4109375 , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.2564791 , 3.50654   ,
        0.        ],
       [0.39788723, 0.3272253 , 1.2096797 , ..., 0.8716151 , 0.5301934 ,
        0.        ]], dtype=float32)>

In [ ]:
# TF
# wyświetlenie rozmiarów parametrów warstwy
vars = ml.trainable_variables
print('Parametry własnej warstwy:')
print(vars[0].shape)
print(vars[1].shape)
print(vars[2].shape)
print(len(vars))

Parametry własnej warstwy:
(4, 16)
(4, 16)
(32, 16)
3


Z kolei w PyTorch warstwy są klasami dziedziczącymi po klasie **Module** z pakietu _torch.nn_. Uczalne parametry warstwy definiuje się bezpośrednio w jej konstruktorze, a za działanie klasy odpowiada metoda _forward_.

In [ ]:
# PT
# dziedziczenie po klasie Module w paczce torch.nn
class MyLayerPT(torch.nn.Module):
    # w konstruktorze definiujemy parametry warstwy
    def __init__(self, input_size, num_weights):
        super().__init__()
        self.num_weights = num_weights

        self.w0 = torch.randn([input_size, self.num_weights], dtype=torch.float32)
        self.w1 = torch.nn.Parameter(torch.randn([input_size, self.num_weights], dtype=torch.float32))
        self.w2 = torch.nn.Parameter(torch.randn([input_size, self.num_weights], dtype=torch.float32))
        self.w3 = torch.nn.Parameter(torch.randn([self.num_weights * 2, self.num_weights], dtype=torch.float32))

    # definicja inferencji modelu
    def forward(self, inputs):
        x1 = torch.nn.functional.relu(inputs @ self.w1)
        x2 = torch.nn.functional.relu(inputs @ self.w2)
        return torch.nn.functional.relu(torch.concat([x1, x2], axis=1) @ self.w3)

In [ ]:
ml = MyLayerPT(4, 16)
dict(ml.named_parameters()).keys()

dict_keys(['w1', 'w2', 'w3'])

In [ ]:
# PT
# deklaracja i inicjalizacja
ml = MyLayerPT(4, 16)

# pierwsze wywołanie
ml(torch.randn([128, 4]))

tensor([[ 0.0000,  0.0000,  0.9941,  ...,  0.8922,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  5.0691,  0.0000,  0.0000],
        [ 0.0000,  4.2725,  0.0000,  ..., 19.0274,  1.2962,  5.2621],
        ...,
        [ 0.0000,  4.3173,  1.6471,  ...,  1.7163,  2.4303,  0.0000],
        [ 0.0000,  6.1239,  0.0000,  ..., 17.6951,  3.6171,  0.0395],
        [ 2.9213,  0.0000,  0.0000,  ...,  2.3566,  0.0000,  0.0000]],
       grad_fn=<ReluBackward0>)

In [ ]:
# PT
# wyświetlenie rozmiarów parametrów warstwy
vars = list(ml.parameters())
print('Parametry własnej warstwy:')
print(vars[0].shape)
print(vars[1].shape)
print(vars[2].shape)
print(len(vars))

Parametry własnej warstwy:
torch.Size([4, 16])
torch.Size([4, 16])
torch.Size([32, 16])
3


Bardzo ważne jest, aby rozdzielać kod na osobne warstwy, ze względu na  możliwość utrzymywanie czytelnego i zdatnego do ponownego użycia kodu.

## Modele

Pojęcie **warstwy** w bibliotece TensorFlow jest bardzo często mylone z **modelami**. Granica pomiędzy nimi jest bardzo cienka, a nawet ich wykorzystanie (głównie w terminologii) jest zamienne. Przyjmuje się, że **modelem są konstrukcje docelowe** korzystające (potencjalnie wielokrotnie) z **warstw** neuronowych. Dynamiczny rozwój badań związanych z sieciami neuronowymi wymusił jednak sytuację, gdzie modele mogą zawierać w sobie inne modele. Przyjmujemy, że **warstwą** powinien być **każdy moduł neuronowy niezależny od jego końcowego zastosowania**.

Jednym z prostszych modeli neuronowych jest model sekwencyjnego przetwarzania, który wywołuje kolejne warstwy (modele) przekazując wyjścia na wejścia kolejnych modeli

In [ ]:
# TF
model = tf.keras.Sequential([
    MyLayerTF(128),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(10)
])

Modele można inicjalizować tak samo jak warstwy - poprzez automatyczną inicjalizację lub ręcznie.

In [ ]:
# TF
model.build([None, 128])

Również analogicznie, możemy pobierać parametry modelu korzystając z **trainable_variables**. Parametrami modelu są wszystkie, rekurencyjnie pobrane, parametry wszystkich warstw i modeli zawartych w danym modelu.

W powyższym przykładzie parametrami modelu będą zarówno parametry warstwy o rozmiarze 128 jak i 10. Modele dziedziczące po klasie **Model** mogą być podsumowane za pomocą specjalnej funkcji:

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_layer_tf_1 (MyLayerTF)       │ (None, 128)            │        65,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,826 (261.04 KB)

 Trainable params: 66,826 (261.04 KB)

 Non-trainable params: 0 (0.00 B)

Aby stworzyć własny model należy analogicznie zadeklarować nową klasę, dziedziczącą tym razem po klasie **Model** dostępnej w paczce *tf.keras*. Ponieważ zarówno **Model** jak i **Layer** dziedziczą ten sam interfejs, również ~~możemy w przypadku modeli definiować inicjalizację modelu w metodzie *build*~~ od pojawienia się Keras 3 (od Tensorflow 2.16) musimy ręcznie definiować definiować metodę *build* lub godzić się na konieczność inicjalizacji przy pierwszym wywołaniu ([link](https://github.com/keras-team/keras/issues/19535#issuecomment-2060299275)).

In [ ]:
# TF
class MyModel(tf.keras.Model):
    def __init__(self, num_weight):
        super().__init__()
        self.ml1 = MyLayerTF(num_weight)
        self.ml2 = MyLayerTF(num_weight)
        self.y = tf.keras.layers.Dense(10)

    def build(self, input_shape):
        self.ml1.build(input_shape)
        input_shape = self.ml1.compute_output_shape(input_shape)
        self.ml2.build(input_shape)
        input_shape = self.ml2.compute_output_shape(input_shape)
        self.y.build(input_shape)

        # lub
        # self.call(tf.random.normal([1, input_shape[1]]))

        self.built = True

    def call(self, inputs, training=None, **kwargs):
        x1 = self.ml1(inputs)
        x2 = self.ml2(x1)
        return self.y(tf.concat([x1, x2], 1))

mm = MyModel(16)
mm.build([None, 128])
#print(mm(tf.random.normal([1, 128])))
mm.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_layer_tf_2 (MyLayerTF)       │ (None, 16)             │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_tf_3 (MyLayerTF)       │ (None, 16)             │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,802 (22.66 KB)

 Trainable params: 5,802 (22.66 KB)

 Non-trainable params: 0 (0.00 B)

W przypadku PyTorch na poziomie kodu nie ma różnicy pomiędzy warstwami a modelami, modele tak samo jak warstwy dziedziczą po klasie **Module** z paczki _torch.nn_.

Tak samo jak w TensorFlow najprostszym modelem jest model sekwencyjny wywołujący po kolei sekwencję warstw (lub modeli).

In [ ]:
# PT
model = torch.nn.Sequential(
    MyLayerPT(128, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)
)

In [ ]:
for param in model[0].parameters():
    param.requires_grad = False

Modele w PyTorch nie posiadają analogicznej metody _summary_, jak ich odpowiedniki w TensorFlow. Aby osiągnąć podobny rezultat można skorzystać z zewnętrznej biblioteki np. _torchinfo_.

In [ ]:
!pip install torchinfo

In [ ]:
# PT
from torchinfo import summary

summary(model, input_size=[1, 128])

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 10]                   --
├─MyLayerPT: 1-1                         [1, 128]                  (65,536)
├─ReLU: 1-2                              [1, 128]                  --
├─Linear: 1-3                            [1, 10]                   1,290
Total params: 66,826
Trainable params: 1,290
Non-trainable params: 65,536
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.27
Estimated Total Size (MB): 0.27

Aby stworzyć własny model należy zadeklarować nową klasę dziedziczącą po klasie **Module**. Analogicznie jak w przypadku warstwy trzeba zaimplementować tworzenie parametrów w konstruktorze oraz zachowanie modelu w metodzie _forward_.

In [ ]:
# PT
class MyModel(torch.nn.Module):
    def __init__(self, input_size, num_weights):
        super().__init__()
        self.ml1 = MyLayerPT(input_size, num_weights)
        self.ml2 = MyLayerPT(num_weights, num_weights)
        self.relu = torch.nn.ReLU()
        self.y = torch.nn.Linear(num_weights * 2, 10)

    # definicja inferencji modelu
    def forward(self, inputs):
        x1 = self.ml1(inputs)
        x2 = self.ml2(x1)
        x = torch.concat([x1, x2], 1)
        x = self.relu(x)
        return self.y(x)

mm = MyModel(128, 16)
summary(mm, input_size=[1, 128])

Layer (type:depth-idx)                   Output Shape              Param #
MyModel                                  [1, 10]                   --
├─MyLayerPT: 1-1                         [1, 16]                   4,608
├─MyLayerPT: 1-2                         [1, 16]                   1,024
├─ReLU: 1-3                              [1, 32]                   --
├─Linear: 1-4                            [1, 10]                   330
Total params: 5,962
Trainable params: 5,962
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.02
Estimated Total Size (MB): 0.02

## Funkcje straty

W bibliotece TensorFlow istnieją dwa schematy definiowania funkcji straty. Ze względu na fakt, że funkcja straty nie jest obiektem posiadającym parametry uczalne (choć w ogólności może posiadać swoje parametry), najczęściej definiowana jest jako osobna funkcja.

Aby przybliżyć potrzebę definiowana osobnych funkcji wprowadźmy przykład w którym zajmujemy się przewidywaniem uzyskanej **średniej oceny** z egzaminu dla dowolnej liczby grup studentów. Posiadamy model który **dla każdego studenta, stara się przewidzieć jego ocenę z egzaminu** (pomijamy informacje na których bazuje model). Jedyne dostępne dane, które posiadamy, to średnia dla każdej z grup. Aby uczyć model przewidywania oceny z egzaminu potrzebujemy najpierw policzyć średnią dla każdej z grupy, a następnie obliczyć docelową funkcję straty.

In [ ]:
# TF
# oceny uczniów (wyjście z modelu predykcyjnego)
y_pred_elements = tf.constant([4.5, 4.0, 2.0, 3.0, 5.0], dtype=tf.float32)
# przypisania uczniów do grup
y_pred_sections_ids = tf.constant([0, 0, 1, 1, 1], dtype=tf.int32)
# znane średnie grup
y_true_sections = tf.constant([4.5, 3.33], dtype=tf.float32)

In [ ]:
# TF
# dokumentacja funkcji scatter_nd: https://www.tensorflow.org/api_docs/python/tf/scatter_nd

def gather_loss(y_pred_elements, y_pred_sections_ids, y_true_sections):
    # obliczenie sumy ocen dla grup
    y_pred_sections = tf.scatter_nd(y_pred_sections_ids[:, tf.newaxis], y_pred_elements, tf.shape(y_true_sections))
    # obliczenie liczby studentów w każdej grupie
    y_pred_counts = tf.scatter_nd(y_pred_sections_ids[:, tf.newaxis], tf.ones_like(y_pred_elements, dtype=tf.float32), tf.shape(y_true_sections))
    # średnia przewidywanych ocen dla każdej grupy
    y_pred_sections = tf.math.divide_no_nan(y_pred_sections, y_pred_counts)
    # funkcja straty
    return tf.reduce_mean(tf.keras.losses.MSE(y_true_sections, y_pred_sections))

In [ ]:
# TF
gather_loss(y_pred_elements, y_pred_sections_ids, y_true_sections)

<tf.Tensor: shape=(), dtype=float32, numpy=0.03125555440783501>

Podobnie w PyTorch.

In [ ]:
# PT
y_pred_elements = torch.tensor([4.5, 4.0, 2.0, 3.0, 5.0], dtype=torch.float32)
# Przypisania uczniów do grup
y_pred_sections_ids = torch.tensor([0, 0, 1, 1, 1], dtype=torch.long)
# Znane średnie grup
y_true_sections = torch.tensor([4.5, 3.33], dtype=torch.float32)

In [ ]:
# PT

def gather_loss_pytorch(y_pred_elements, y_pred_sections_ids, y_true_sections):
    # Utworzenie pustych tensorów o kształcie tensora wyjściowego do przechowywania sum i zliczeń
    y_pred_sections = torch.zeros_like(y_true_sections)
    y_pred_counts = torch.zeros_like(y_true_sections)

    # Obliczenie sumy ocen dla każdej grupy za pomocą `index_add_`
    # (odpowiednik `tf.scatter_nd` dla operacji sumowania)
    y_pred_sections.index_add_(0, y_pred_sections_ids, y_pred_elements)

    # Obliczenie liczby studentów w każdej grupie
    y_pred_counts.index_add_(0, y_pred_sections_ids, torch.ones_like(y_pred_elements))

    # Obliczenie średniej przewidywanych ocen dla każdej grupy
    # Zabezpieczenie przed dzieleniem przez zero
    y_pred_sections = y_pred_sections / y_pred_counts
    y_pred_sections[y_pred_counts == 0] = 0 # Ręczne zastąpienie NaN wartością 0

    # Funkcja straty (Mean Squared Error)
    return F.mse_loss(y_pred_sections, y_true_sections)

In [ ]:
gather_loss_pytorch(y_pred_elements, y_pred_sections_ids, y_true_sections)

tensor(0.0313)

Ponowne wykorzystanie funkcji strraty jest potrzebne podczas:
- optymalizacji modelu przetwarzania sieci neuronowej,
- walidacji (doboru parametrów),
- testowania

## Prototypowanie

Jedną z ważniejszych cznności podczas prowadzenia prac z sieciami neuronowymi, **szczególnie w kontekście projektowania własnych rozwiązań** jest **prototypowanie**.

Prototypowanie polega (najczęściej) na :
- określeniu specyfikacji danych wejściowych i wyjściowych,
- utworzeniu przykładowych danych, o danej specyfikacji,
- tworzeniu i dostrajaniu parametrów warstw i modeli niezbędnych do osiągnięcia danych wyjściowych

### Przykład użycia 1

Posiadasz zbiór próbek skomplikowanej funkcji dwuwymiarowej, której definicji nie znasz, ale potrzebujesz wiedzieć jaką wartość będzie przyjmowała dla argumentów wejściowych innych niż te zawarte w próbce.

Dane wejściowe:
- funkcja dwuargumentowa: wektor 2-elementowy

Dane wyjściowe:
- pojedyncza wartość: wektor 1-elementowy

Rozpoznana dziedzina problemu:
- uczenie nadzorowane,
- problem regresji

In [ ]:
# próbki danych i odpowiadające im rozkłady prawdopodobieństw
x = ...
y_true = ...


# definicja i inicjalizacja modelu neuronowego
model = ...


# definicja funkcji straty
def loss_fn(y_true, y_pred):
    pass


# wywołanie modelu i obliczenie fukcji straty
# y_pred = model(x)
# loss = loss_fn(y_pred, y_true)

In [ ]:
# PT

# próbki funkcji jako dane wejściowe i odpowiadające im wartości funkcji
x = torch.rand(16, 2)
y_true = torch.rand(16, 1)

# definicja i inicjalizacja modelu neuronowego
model = nn.Sequential(
    nn.Linear(2, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)

# definicja funkcji straty
loss_fn = nn.MSELoss()

# wywołanie modelu i obliczenie fukcji straty
y_pred = model(x)
loss = loss_fn(y_pred, y_true)

print(loss)

tensor(0.4744, grad_fn=<MseLossBackward0>)


In [ ]:
# TF

# próbki funkcji jako dane wejściowe i odpowiadające im wartości funkcji
x = tf.random.uniform([16, 2])
y_true = tf.random.uniform([16, 1])

# definicja i inicjalizacja modelu neuronowego
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1)
])


# definicja funkcji straty
def loss_fn(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.MSE(y_true, y_pred))


# wywołanie modelu i obliczenie fukcji straty
y_pred = model(x)
loss = loss_fn(y_pred, y_true)

print(loss)

tf.Tensor(0.21790062, shape=(), dtype=float32)


### Przykład użycia 2

Zajmujesz się problematyką klasyfikacji obrazów kolorowych. Posiadasz zbiór obrazów w wymiarach **H x W** w przestrzeni **RGB** (wartości z przedziału 0-1). Twoim zadaniem jest odgadnięcie jednej z 10 klas obrazów.

Dane wejściowe:
- obrazy o stałym rozmiarze **H x W**,
- obrazy w przestrzeni **RGB** (3 wartości),
- piksele znormalizowane do wartości 0-1,

Dane wyjściowe:
- rozkład prawdopodobieństwa dla 10 klas: wektor 10-elementowy

Rozpoznana dziedzina problemu:
- uczenie nadzorowane,
- problem klasyfikacji


In [ ]:
# próbki danych jako obrazy i odpowiadające im rozkłady prawdopodobieństw (wektory 10-elmentowe)
x = ...
y_true = ...


# definicja i inicjalizacja modelu neuronowego
model = ...


# definicja funkcji straty
def loss_fn(y_true, y_pred):
    pass


# wywołanie modelu i obliczenie fukcji straty
# y_pred = model(x)
# loss = loss_fn(y_pred, y_true)

In [ ]:
#@title Schowane

# Definicja wymiarów dla przejrzystości
BATCH_SIZE = 16
H, W = 32, 32  # Przykładowa wysokość i szerokość obrazu
NUM_CLASSES = 10 # Liczba klas (np. dla MNIST lub CIFAR-10)

# próbki danych jako obrazy (B x C x H x W) i odpowiadające im etykiety klas (wektory indeksów)
# W PyTorch standardowym formatem dla obrazów jest (Batch, Kanały, Wysokość, Szerokość)
x = torch.rand(BATCH_SIZE, 3, H, W)
# Dla CrossEntropyLoss, etykiety to zazwyczaj wektor indeksów klas, a nie wektory one-hot
y_true = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,))


# definicja i inicjalizacja modelu neuronowego - prosta sieć konwolucyjna (CNN)
model = nn.Sequential(
    # Blok 1
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), # Rozmiar obrazu: 32x32 -> 16x16

    # Blok 2
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), # Rozmiar obrazu: 16x16 -> 8x8

    # Spłaszczenie wyniku z warstw konwolucyjnych do wektora
    nn.Flatten(),

    # Warstwy w pełni połączone (klasyfikator)
    # 32 kanały * 8x8 rozmiar mapy cech po poolingach
    nn.Linear(32 * 8 * 8, 128),
    nn.ReLU(),
    nn.Linear(128, NUM_CLASSES) # Warstwa wyjściowa dająca logity dla 10 klas
)


# definicja funkcji straty - CrossEntropyLoss dla klasyfikacji wieloklasowej
# Ta funkcja w PyTorch automatycznie stosuje Softmax na wyjściu z modelu
loss_fn = nn.CrossEntropyLoss()


# wywołanie modelu i obliczenie fukcji straty
y_pred = model(x) # Wyjście z modelu to tzw. "logity" (surowe wyniki)
loss = loss_fn(y_pred, y_true)

# Wypisanie obliczonej straty
print(loss)

### Przykład użycia 3

Podczas pracy nad przetwarzaniem obrazów o wysokiej rozdzielczości (w przestrzeni RGB) natrafiasz na problem nieustandaryzowanego zbioru danych, którego opracowanie było bardzo kosztowne i niemożliwe do powtórzenia w najbliższym czasie. Do pracy potrzebujesz danych o jak najwyższej rozdzielczości natomiast zbiór danych który posiadasz zawiera obrazy **w różnych (niesatysfakcjonujących Cię) rozdzielczościach**. Widząc niesamowite możliwości sieci neuronowych postanawiasz opracować sieć neuronową, która **(1) pobiera obrazy wejściowe i zwraca obrazy w jednej, ustalonej rozdzielczości** oraz **(2) obsługuje obrazy wejściowe w różnych rozdzielczościach**.

Dane wejściowe:
- obrazy o zmiennym rozmiarze,
- obrazy w przestrzeni RGB (3 wartości)

Dane wyjściowe:
- obraz o stałym rozmiarze (parametr),
- obraz w przestrzeni RGB (3 wartości),

Rozpoznana dziedzina problemu:
- uczenie ???,
- problem ???

In [ ]:
# próbki danych jako obrazy (h x w x 3) i odpowiadające im rozkłady prawdopodobieństw (wektory 10-elmentowe)
x = ...
y_true = ...


# definicja i inicjalizacja modelu neuronowego
model = ...


# definicja funkcji straty
def loss(y_true, y_pred):
    pass


# wywołanie modelu i obliczenie fukcji straty
# loss(y_true, model(x))

# Uczenie modeli neuronowych

Modele neuronowe, zaimplementowane w Kerasie, można uczyć na dwa główne sposoby:

- korzystając z frameworku dostępnego poprzez interfejs klasy Model
- ręcznie projektując funkcję straty, optymalizatory, obliczanie gradientu oraz jego aplikację do algorytmu uczącego

Pierwsze podejście jest podejściem często stosowanym w pracy inżynieryjnej przy sprawdzonych i gotowych modelach neuronowych, których architektury zostały zweryfikowane i zoptymalizowane pod kątem nauki na dostępnym sprzęcie (np. wiele modułów GPU, klastry, itp.).

Podejściem bardziej zaawansowanym, a także trudniejszym w implementacji i zrozumieniu, jest ręczna implementacja procesu uczenia. Olbrzymią zaletą tej drugiej jest możliwość korzystania z nieszablonowych metod uczenia oraz sposobność dostępu do wyników pośrednich, dzięki którym można wspomóc proces prototypowania (np. wychwycając problemy błędów numerycznych - szczególnie powszechnych w uczeniu ze wzmocnieniem, lub błędów związanych z różniczkowalnością modeli).

Poniżej przedstawiony został przykład prototypowania, w którym dla pewnego wektora wejściowego o zadanym rozmiarze i odpowiadającmu mu wektorowi docelowemu o ustalonym rozmiarze, wykonana została regresja razem z krokiem uczenia i inferencji.

Implementację nauki modelu neuronowego można zacząć od wprowadzonego powyżej etapu prototypowania:

In [ ]:
# TF

# rozmiar wektora wejściowego i wyjściowego
batch_size = 32
in_vec = 128
out_vec = 16

# przykładowe dane wejściowe i wyjściowe
x = tf.random.uniform([batch_size, in_vec], dtype=tf.float32)
y_true = tf.random.uniform([batch_size, out_vec], dtype=tf.float32)

# neuronowy model przetwarzania
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(out_vec)
])

# funkcja straty dla modelu i danych
def loss_fn(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.MSE(y_true, y_pred))

Dobrą praktyką jest wyszczególnienie etapu **inferencji** (odpytania) modelu w osobnej funkcji w celach późniejszej optymalizacji.

In [ ]:
# TF
def query(x, training=False):
    return model(x, training=training)

Do nauki modelu potrzebny jest obiekt klasy implementującej algorytm **wstecznej propagacji błędu**. Paczką w bibliotece TensorFlow zawierającą różne implementacje tego algorytmu jest *tf.optimizers*. Klasy te dziedziczą po tej samej klasie optymalizatora i mogą być (w znacznej większości) wykorzystywane zamiennie.

In [ ]:
# TF
# podstawowy algorytm Stochastic Gradient Descent
optimizer = tf.optimizers.SGD(0.001)

Ostatnim etapem jest utworzenie kroku uczenia modelu (wersja zaawansowana). Aby to zrobić, należy wykorzystać mechanizm **tf.GradientTape**, który pozwala na **automatyczne rejestrowanie gradientów wszystkich różniczkowalnych operacji we wskazanym obszarze**.

Wewnątrz wskazanego przez GradientTape obszaru powinny być wykonane wszelkie operacje prowadzące do obliczenia **końcowej wartości funkcji straty**. Ważne jest, aby ostateczny tensor reprezentujący wartość funkcji straty nie był modyfikowany poza wskazanym obszarem przed **pobraniem pochodnych funkcji straty po wskazanych zmiennych**.

Zmienne które wskazujemy zazwyczaj są zmiennymi modelu neuronowego i mogą zostać pobrane tak samo jak dla każdego modelu w TensorFlow - poprzez property **trainable_variables**.

Aby pobrać pochodne należy wywołać funkcję **gradient** z poziomu obiektu GradientTape.

Ostatnim krokiem jest zaaplikowanie wyliczonych pochodnych do algorytmu uczenia (metoda _apply_gradients_ z interfejsu klasy optymalizatora).

In [ ]:
# TF
def train(x, y_true):
    # utworzenie mechanizmu zapisywania gradientów dla operacji różniczkowalnych
    with tf.GradientTape() as tape:
        # wywołanie modelu dla danych wejściowych (pamiętając o fladze training=True)
        y_pred = query(x, True)
        # obliczenie funkcji straty, która poza obszarem GradientTape i przed .gradients() nie jest już modyfikowana
        loss = loss_fn(y_true, y_pred)

    # pobranie zmiennych względem których obliczane są gradienty
    vars = model.trainable_variables
    grads = tape.gradient(loss, vars)
    # aplikacja gradientów do optymalizatora
    grads = optimizer.apply_gradients(zip(grads, vars))

    return loss

Dla tak zdefiniowanego modelu przetwarzania można przeprowadzić testową pętlę uczenia razem ze zmierzeniem czasów przetwarzania.


In [ ]:
# TF
# liczba kroków aby uzyskać wiarygodne czasy przetwarzania
num_runs = 100

# niektóre modele potrzebują więcej czasu podczas inicjalizacji
# na alokowanie dodatkowych struktur (szczególnie gdy wykorzystuja się GPU)
query(x)
train(x, y_true)

# zmierzenie czasu działania odpytania
start = time.time()
for i in range(num_runs):
    query(x)
end = time.time()

print('Średni czas odpytania modelu neuronowego (batch):', (end - start) / num_runs)
print('Średni czas odpytania modelu neuronowego (obiekt):', (end - start) / (num_runs * batch_size), '\n')

# zmierzenie czasu działania kroku uczenia
start = time.time()
for i in range(num_runs):
    train(x, y_true)
end = time.time()

print('Średni czas uczenia modelu neuronowego (batch):', (end - start) / num_runs)
print('Średni czas uczenia modelu neuronowego (obiekt):', (end - start) / (num_runs * batch_size))

Średni czas odpytania modelu neuronowego (batch): 0.00538576602935791
Średni czas odpytania modelu neuronowego (obiekt): 0.0001683051884174347 

Średni czas uczenia modelu neuronowego (batch): 0.026178081035614014
Średni czas uczenia modelu neuronowego (obiekt): 0.0008180650323629379


Analogiczny proces możemy również przeprowadzić przy użyciu PyTorch. Zaczynamy od utworzenia modelu.

In [ ]:
# PT
# przykładowe dane wejściowe i wyjściowe
x = torch.rand([batch_size, in_vec], dtype=torch.float32)
y_true = torch.rand([batch_size, out_vec], dtype=torch.float32)

# neuronowy model przetwarzania
model = torch.nn.Sequential(
    torch.nn.Linear(in_vec, 64),
    torch.nn.BatchNorm1d(64),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(64, 32),
    torch.nn.BatchNorm1d(32),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(32, out_vec)
)

# funkcja straty dla modelu i danych
def loss_fn(y_true, y_pred):
    return torch.nn.functional.mse_loss(y_true, y_pred)

Do optymalizacji parametrów modelu ponownie wykorzystamy algorytm **Stochastic Gradient Descent**. W przypadku PyTorch parametry, które mają być poddane uczeniu, należy podać bezpośrednio w wywołaniu konstruktora.

In [ ]:
# PT
# podstawowy algorytm Stochastic Gradient Descent
optimizer = torch.optim.SGD(model.parameters(), 0.001)

W przypadku PyTorch nie ma potrzeby wskazywania obszaru, wewnątrz którego są wykonane operacje prowadzące do obliczenia wartości funkcji straty.

Obliczanie gradientów odbywa się poprzez wywołanie metody _backward_ bezpośrednio na wartości funkcji straty.


Ostatnim krokiem jest zaaplikowanie wyliczonych pochodnych do algorytmu uczenia (metoda _step_ z interfejsu klasy optymalizatora).

In [ ]:
x = torch.randn(10)
x

tensor([ 1.2524, -0.2913, -0.4424, -0.6277, -0.9834,  0.6896, -1.2686,  0.7244,
         0.3296,  0.5962])

In [ ]:
d = nn.Dropout(p=0.5)
d.eval()
d(x)

tensor([ 1.2524, -0.2913, -0.4424, -0.6277, -0.9834,  0.6896, -1.2686,  0.7244,
         0.3296,  0.5962])

In [ ]:
model.train()
# wyzerowanie poprzednio zapisanego gradientu
optimizer.zero_grad()
# wywołanie modelu dla danych wejściowych
y_pred = model(x)
# obliczenie funkcji straty
loss = loss_fn(y_true, y_pred)

In [ ]:
loss.backward()

In [ ]:
list(model.parameters())[0].grad

tensor([[-0.0065,  0.0044, -0.0136,  ..., -0.0010,  0.0101, -0.0005],
        [-0.0178, -0.0173, -0.0022,  ...,  0.0088, -0.0012, -0.0301],
        [-0.0020, -0.0089, -0.0060,  ...,  0.0005, -0.0093, -0.0073],
        ...,
        [-0.0015, -0.0129,  0.0069,  ..., -0.0012,  0.0014,  0.0067],
        [-0.0031, -0.0083, -0.0091,  ...,  0.0116, -0.0027,  0.0092],
        [-0.0032, -0.0126, -0.0095,  ..., -0.0119,  0.0021,  0.0032]])

In [ ]:
# PT
def train(x, y_true):
    # przestawienie modelu w tryb uczenia
    model.train()
    # wyzerowanie poprzednio zapisanego gradientu
    optimizer.zero_grad()
    # wywołanie modelu dla danych wejściowych
    y_pred = model(x)
    # obliczenie funkcji straty
    loss = loss_fn(y_true, y_pred)
    # policzenie gradientu względem funkcji straty
    loss.backward()
    # aktualizacja wag modelu
    optimizer.step()
    return loss

Następnie ponownie mierzymy czasy inferencji i uczenia modelu.

In [ ]:
# PT
# liczba kroków aby uzyskać wiarygodne czasy przetwarzania
num_runs = 100

model(x)
train(x, y_true)

# zmierzenie czasu działania odpytania

# torch.cuda.synchronize()
start = time.time()
for i in range(num_runs):
    model(x)
#   torch.cuda.synchronize()
end = time.time()

print('Średni czas odpytania modelu neuronowego (batch):', (end - start) / num_runs)
print('Średni czas odpytania modelu neuronowego (obiekt):', (end - start) / (num_runs * batch_size), '\n')

# zmierzenie czasu działania kroku uczenia
start = time.time()
for i in range(num_runs):
    train(x, y_true)
end = time.time()

print('Średni czas uczenia modelu neuronowego (batch):', (end - start) / num_runs)
print('Średni czas uczenia modelu neuronowego (obiekt):', (end - start) / (num_runs * batch_size))

Średni czas odpytania modelu neuronowego (batch): 0.0002991604804992676
Średni czas odpytania modelu neuronowego (obiekt): 9.348765015602112e-06 

Średni czas uczenia modelu neuronowego (batch): 0.0008342194557189941
Średni czas uczenia modelu neuronowego (obiekt): 2.6069357991218565e-05


# Optymalizacja przetwarzania neuronowego

TensorFlow posiada dwie opcje wykonania kodu:
- eager execution - standardowe wykonanie sekwencyjne "linijka po linijce", możliwe do debugowania,
- graph execution - utowrzenie niskopoziomowego grafu definiującego kolejność i zależność wykonanych operacji,

Eager execution jest domyślnie wykorzystystywanym trybem działania TensorFlow (od wersji 2.0) i idealnie nadaje się do prototypowania i debugowania kodu.

Graph execution natomiast jest zaawansowanych mechanizmem, który pozwala na znaczną optymalizację modelu jednak w zamian traci się możliwość sprawnego debugowania kodu oraz **ograniczona jest możliwość wykorzystywania jakichkolwiek innych bibliotek niż TensorFlow**.

Aby stworzyć graf przetwarzania w TensorFlow należy wybrany fragment kodu umieścić w osobnej funkcji i dodać do niej dekorator **@tf.function**, lub bezpośrednio skorzystać z funkcji **tf.function** tworząc osobną instancję funkcji.

In [ ]:
# funkcja operation1 jest zastąpiona przez definicję grafu
@tf.function
def operation1(a, b):
    return tf.reduce_mean(a) + tf.reduce_mean(b)

# przykładowe dane wejściowe
a = tf.random.uniform([128, 1024], dtype=tf.float32)
b = tf.random.uniform([128, 2048], dtype=tf.float32)

# uruchomienie grafu
c = operation1(a, b)

print('Rezultat działania operacji 1:', c.numpy())

Rezultat działania operacji 1: 0.9989994


Warto zwrócić uwagę na mechanizm **automatycznego tworzenia instancji grafu przez TensorFlow**. Korzystając z tego mechanizmu, TensorFlow przetrzymuje w pamięci wiele instancji grafu i wywołuje te które pasują do danych wejściowych. W przypadku, gdy żaden graf nie pasuje do danych wejściowych, **następuje próba utworznie nowego grafu**.

Bardzo często, gdy argumentami wejściowymi są dane całkowitoliczbowe, których wartość powoduje inne schematy działania modelu, niezbędna jest ręczna inicjalizacja grafu przetwarzania. W innym przypadku, możliwe jest, że co wywołanie TensorFlow będzie próbował tworzyć nowy graf.

Podczas debugowania kodu należy szczególnie uważać na grafy TensorFlow. Każdokrotnie podczas inicjalizowania funkcji nastąpi wywołanie schematu przetwarzania z pustymi tensorami o ustalonych rozmiarach.

In [ ]:
# definicja grafu
@tf.function
def operation2(a, b):
    print('Nowa instancja grafu!')
    print('Rozmiar parametru a:', a)
    print('Rozmiar parametru b:', b)
    return tf.reduce_mean(a) + tf.reduce_mean(b)

# inicjalizacja danych, grafu i uruchomienie go
a = tf.random.uniform([10, 128], dtype=tf.float32)
b = tf.random.uniform([10, 256], dtype=tf.float32)

# utworzony zostanie nowy graf
print(operation2(a, b))

# inicjalizacja danych, grafu i uruchomienie go
a = tf.random.uniform([20, 128], dtype=tf.float32)
b = tf.random.uniform([20, 256], dtype=tf.float32)

# utworzony zostanie nowy graf
print(operation2(a, b))

# wykorzystany zostanie graf posiadający sygnaturę [20, 128], [20, 256]
print(operation2(a, b))

Nowa instancja grafu!
Rozmiar parametru a: Tensor("a:0", shape=(10, 128), dtype=float32)
Rozmiar parametru b: Tensor("b:0", shape=(10, 256), dtype=float32)
tf.Tensor(0.9968965, shape=(), dtype=float32)
Nowa instancja grafu!
Rozmiar parametru a: Tensor("a:0", shape=(20, 128), dtype=float32)
Rozmiar parametru b: Tensor("b:0", shape=(20, 256), dtype=float32)
tf.Tensor(1.0050354, shape=(), dtype=float32)
tf.Tensor(1.0050354, shape=(), dtype=float32)


Przetwarzania grafowe pozwala na wykonanie wielu operacji równolegle, zgodnie z ich grafem przetwarzania, a nie z sekwencją wykonania. Takie działanie może spowodować drastyczny wzrost wydajności.

In [ ]:
# TF
# liczba przebiegów testowych
num_runs = 100

# definicja schematu przetwarzania
def operation3(a):
    return tf.add_n([tf.reduce_mean(a) for i in range(100)])

# ręczna definicja grafu i ręczna inicjalizacja grafu z podanym rozmiarem wejściowym
operation3_graph = tf.function(operation3)
operation3_graph = operation3_graph.get_concrete_function(tf.TensorSpec([128, 1024], dtype=tf.float32))

# przykładowe dane wejściowe
a = tf.random.uniform([128, 1024], dtype=tf.float32)

# przetwarzanie czystego schematu neuronowego w trybie eager execution
start = time.time()
for i in range(num_runs):
    operation3(a)
end = time.time()
eager_time = (end - start) / num_runs

print(f"Średni czas wykonania (Eager): {eager_time:.8f} sekund")


# przetwarzanie grafu
start = time.time()
for i in range(num_runs):
  operation3_graph(a)
end = time.time()
graph_time = (end - start) / num_runs

print(f"Średni czas wykonania (Graph): {graph_time:.8f} sekund")

Średni czas wykonania (Eager): 0.01624136 sekund
Średni czas wykonania (Graph): 0.00057976 sekund


 `torch.compile` (wprowadzony w PyTorch 2.0) jest rzeczywiście nowoczesnym i potężnym odpowiednikiem `tf.function`, który kompiluje kod Pythona do zoptymalizowanego grafu obliczeniowego.

In [ ]:
# PT
# liczba przebiegów testowych
num_runs = 100

# definicja schematu przetwarzania
def operation3(a):
    return torch.sum(torch.stack([torch.mean(a) for _ in range(100)]))

operation3_compiled = torch.compile(operation3, mode="reduce-overhead")


# przykładowe dane wejściowe
a = torch.rand(128, 1024, dtype=torch.float32)


_ = operation3(a)
# torch.cuda.synchronize()

start = time.time()
for _ in range(num_runs):
    operation3(a)
# torch.cuda.synchronize()
end = time.time()
eager_time = (end - start) / num_runs

print(f"Średni czas wykonania (Eager): {eager_time:.8f} sekund")



# WAŻNE: Pierwsze wywołanie skompilowanej funkcji uruchamia proces kompilacji JIT (Just-In-Time).
# Ten krok może zająć chwilę i nie powinien być wliczany do pomiaru wydajności.
_ = operation3_compiled(a)
# torch.cuda.synchronize()

start = time.time()
for _ in range(num_runs):
    operation3_compiled(a)
# torch.cuda.synchronize()
end = time.time()
compiled_time = (end - start) / num_runs

print(f"Średni czas wykonania (Compiled): {compiled_time:.8f} sekund")

Średni czas wykonania (Eager): 0.00183373 sekund
Średni czas wykonania (Compiled): 0.00031240 sekund


# Przykład użycia 4

Stwórz model neuronowy do klasyfikacji obrazów pochodzących ze zbioru MNIST.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LEARNING_RATE = 1e-3
BATCH_SIZE = 128
EPOCHS = 3
VALIDATION_SPLIT = 0.15
MODEL_SAVE_PATH = "mnist_cnn_model.pth"

# Zdefiniowanie transformacji danych
transform = transforms.Compose([
    transforms.ToTensor(), # Konwersja obrazu do tensora
    transforms.Normalize((0.1307,), (0.3081,)) # Normalizacja danych MNIST
])


# Przygotowanie danych (trening, walidacja, test)

# Załadowanie pełnego zbioru treningowego
full_train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

# Podział zbioru treningowego na treningowy i walidacyjny
ds_train, ds_val = random_split(full_train_dataset, [1 - VALIDATION_SPLIT, VALIDATION_SPLIT])

# Załadowanie zbioru testowego
ds_test = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

# Utworzenie DataLoaderów
train_loader = DataLoader(
    dataset=ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    #num_workers=2,
    #pin_memory=True
)

val_loader = DataLoader(
    dataset=ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False, # Zbiór walidacyjny nie musi być mieszany
    #num_workers=2,
    #pin_memory=True
)

test_loader = DataLoader(
    dataset=ds_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    #num_workers=2,
    #pin_memory=True
)

print(f"Liczba próbek treningowych: {len(ds_train)}")
print(f"Liczba próbek walidacyjnych: {len(ds_val)}")
print(f"Liczba próbek testowych: {len(ds_test)}")

Liczba próbek treningowych: 51000
Liczba próbek walidacyjnych: 9000
Liczba próbek testowych: 10000


In [ ]:
# Implementacja modelu (prosta sieć konwolucyjna)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Warstwy konwolucyjne do ekstrakcji cech
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
        )
        # Warstwy w pełni połączone do klasyfikacji
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128), # Obraz 28x28 po 2x MaxPool staje się 7x7
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, 10) # 10 klas wyjściowych (cyfry 0-9)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.shape[0], -1) # Spłaszczenie tensora przed warstwami FC
        x = self.fc_layers(x)
        return x

In [ ]:
model = SimpleCNN().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\nRozpoczęcie uczenia modelu...")

for epoch in range(EPOCHS):
    # Faza uczenia
    model.train() # Ustawienie modelu w tryb treningowy
    train_loss = 0.0
    for images, labels in tqdm(train_loader):
        # Przeniesienie danych na odpowiednie urządzenie (CPU/GPU)
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Faza walidacji
    model.eval() # Ustawienie modelu w tryb ewaluacji
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad(): # Wyłączenie obliczania gradientów w fazie walidacji
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}], "
          f"Train loss: {avg_train_loss:.4f}, "
          f"Val loss: {avg_val_loss:.4f}, "
          f"Val acc: {val_accuracy:.2f}%")


# Zapisanie wytrenowanego modelu
# torch.save(model.state_dict(), MODEL_SAVE_PATH)


Rozpoczęcie uczenia modelu...


100%|██████████| 399/399 [00:49<00:00,  8.09it/s]


Epoch [1/3], Train loss: 0.1058, Val loss: 0.0430, Val acc: 98.60%


100%|██████████| 399/399 [00:52<00:00,  7.61it/s]


Epoch [2/3], Train loss: 0.0353, Val loss: 0.0345, Val acc: 98.89%


100%|██████████| 399/399 [00:51<00:00,  7.80it/s]


Epoch [3/3], Train loss: 0.0230, Val loss: 0.0347, Val acc: 98.87%


In [ ]:
test_loss = 0.0
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test loss: {avg_test_loss:.4f}")
print(f"Test acc: {test_accuracy:.2f}%")

Test loss: 0.0356
Test acc: 98.83%


# Materiały dodatkowe:

- Tutoriale z oficjalnej strony TensorFlow [(link)](https://www.tensorflow.org/tutorials)
- Tutoriale z oficjalnej strony PyTorch [(link)](https://pytorch.org/tutorials/)
- ~~Artykuły naukowe z dostępnymi oficjalnymi implementacjami [(link)](https://paperswithcode.com/)~~
